# Can Low Cardiac Output During Surgery Harm the Kidneys?

Every anaesthesiologist knows that maintaining adequate organ perfusion is one of the central goals of intraoperative care. But how do we know whether the heart is pumping *enough* blood to protect vulnerable organs — particularly the kidneys?

In this notebook, we investigate a deceptively simple clinical question: **does a low cardiac index during surgery increase the risk of acute kidney injury (AKI) afterwards?**

## What you will learn

- **What cardiac index means** and how it is measured in the operating room
- **What AKI is**, how we define it using creatinine, and why it matters for surgical outcomes
- How to extract and link intraoperative haemodynamic data with postoperative laboratory results from 981 surgical cases in VitalDB
- Whether spending more time at a low cardiac index is associated with a higher chance of kidney injury

## Why does this matter?

Acute kidney injury after surgery is common — it affects roughly 5–30% of patients depending on the type of operation, and even mild AKI is associated with longer hospital stays, higher costs, and increased mortality. If we can identify an intraoperative haemodynamic threshold that predicts kidney injury, we might be able to intervene *during* the operation — before the damage is done.

## Key terms we will use

Before we dive in, let us define a few concepts you will encounter throughout this chapter:

- **Cardiac index (CI):** The volume of blood the heart pumps per minute, divided by the patient's body surface area. It is typically measured in litres per minute per square metre (L/min/m²). A normal resting CI is roughly 2.5–4.0 L/min/m². We divide by body surface area so we can fairly compare a small patient with a large one.

- **Acute kidney injury (AKI):** A sudden decline in kidney function, detected by a rise in serum creatinine. We use the internationally recognised **KDIGO** (Kidney Disease: Improving Global Outcomes) criteria, which define Stage 1 AKI as a postoperative creatinine more than 1.5 times the preoperative baseline within 48 hours.

- **Creatinine:** A waste product of muscle metabolism that is filtered by the kidneys. When the kidneys are working well, creatinine stays low. When kidney function drops, creatinine rises — making it a useful (if imperfect) marker of kidney health.

- **Perfusion pressure:** The driving pressure that pushes blood through an organ's capillary bed. For the kidneys, this depends on both blood pressure and cardiac output working together.

Let us begin by loading our data.

In [1]:
#pip install vitaldb
import vitaldb
import pandas as pd
import numpy as np

#df_cases = pd.read_csv("https://api.vitaldb.net/cases")  # Load clinical data
#df_trks = pd.read_csv('https://api.vitaldb.net/trks')  # Load track list
#df_labs = pd.read_csv('https://api.vitaldb.net/labs')  # Load lab result

## What data are we working with?

Our first step is to load the three core data files from VitalDB. These contain:

1. **Track data** — a catalogue of all the physiological signals (waveforms, vital signs) recorded for each case. This is where we will find cardiac index measurements from devices like the Vigileo, EV1000, and Vigilance monitors.
2. **Laboratory data** — blood test results with timestamps, including the creatinine values we need to detect AKI.
3. **Case data** — demographic and surgical information for each of the 6,388 patients.

Let us start by loading and inspecting the track and laboratory catalogues to see what is available.

In [2]:
#df_trks.to_csv("vital_db/tracks.csv", index=False)
#df_labs.to_csv("vital_db/labs.csv", index=False)

In [3]:
tracks = pd.read_csv("vital_db/tracks.csv")
tracks.head()

,caseid,tname,tid
0,1,BIS/BIS,fd869e25ba82a66cc95b38ed47110bf4f14bb368
1,1,BIS/EEG1_WAV,0aa685df768489a18a5e9f53af0d83bf60890c73
2,1,BIS/EEG2_WAV,ad13b2c39b19193c8ae4a2de4f8315f18d61a57e
3,1,BIS/EMG,2525603efe18d982764dbca457affe7a45e766a9
4,1,BIS/SEF,1c91aec859304840dec75acf4a35da78be0e8ef0


### What signals are available?

The tracks table lists every physiological signal recorded for each case — 196 different signal types in total, covering everything from BIS (depth of anaesthesia) to ventilator pressures, ECG waveforms, and the cardiac output measurements we are looking for. Let us see the full list of available signal names.

In [4]:
tracks.tname.unique()

<StringArray>
[            'BIS/BIS',        'BIS/EEG1_WAV',        'BIS/EEG2_WAV',
             'BIS/EMG',             'BIS/SEF',             'BIS/SQI',
              'BIS/SR',          'BIS/TOTPOW',          'Primus/AWP',
          'Primus/CO2',
 ...
  'Orchestra/VASO_VOL', 'Orchestra/DOBU_RATE',  'Orchestra/DOBU_VOL',
  'Orchestra/NPS_RATE',   'Orchestra/NPS_VOL',  'Orchestra/VEC_RATE',
   'Orchestra/VEC_VOL',     'Solar8000/ST_V5',  'Orchestra/AMD_RATE',
   'Orchestra/AMD_VOL']
Length: 196, dtype: str

That is a lot of signals! Among them, you will spot names like `Vigileo/CI`, `EV1000/CI`, and `Vigilance/CI` — these are cardiac index measurements from three different cardiac output monitoring devices commonly used in operating theatres. We will use all three to maximise our sample size.

Now let us look at the laboratory data — specifically, what blood tests are recorded.

In [5]:
labs = pd.read_csv("vital_db/labs.csv")
labs.head()

,caseid,dt,name,result
0,1,594470,alb,2.9
1,1,399575,alb,3.2
2,1,12614,alb,3.4
3,1,137855,alb,3.6
4,1,399575,alt,12.0


Each row in the labs table represents a single blood test result: a `caseid` (which patient), a `dt` (timestamp in seconds — negative values mean "before surgery", positive values mean "after the start of anaesthesia"), a `name` (the test abbreviation), and a `result` (the numeric value). Let us see all the available test types.

In [6]:
labs.name.unique()

<StringArray>
[  'alb',   'alt',  'aptt',   'ast',   'bun',    'cl',    'cr',   'crp',
   'fib',   'gfr',  'gluc',    'hb',  'hco3',   'hct',   'ica',     'k',
   'lac',    'na',  'pco2',    'ph',   'plt',   'po2',   'pt%', 'ptinr',
 'ptsec',  'sao2',  'tbil', 'tprot',   'wbc',    'be',     'p',   'esr',
   'ccr',  'ammo']
Length: 34, dtype: str

### Which lab tests do we have?

The laboratory dataset contains 34 different blood tests. The one we care about most is **`cr`** — serum creatinine. This is the marker we will use to detect AKI by comparing the preoperative value (before surgery) with the postoperative value (within 48 hours after surgery).

You can also see other familiar tests: `hb` (haemoglobin), `k` (potassium), `lac` (lactate), `gfr` (glomerular filtration rate), and many more. For now, creatinine is our focus.

## Which patients had cardiac output monitoring?

Not every patient in VitalDB had a cardiac output monitor during their surgery — these devices are typically reserved for higher-risk cases where we expect significant haemodynamic shifts (major abdominal surgery, cardiac surgery, patients with significant cardiac disease, etc.).

We need to find all cases where a cardiac index signal was recorded by *any* of the three monitor types: **Vigileo**, **EV1000**, or **Vigilance**. These are all pulse-contour or thermodilution-based devices that estimate how much blood the heart pumps each minute, normalised to body size.

Let us identify those cases now.

In [7]:
caseids = list(
    set(tracks.loc[tracks.tname == "Vigileo/CI", 'caseid']) |
    set(tracks.loc[tracks.tname == 'EV1000/CI', 'caseid'])  |
    set(tracks.loc[tracks.tname == 'Vigilance/CI', 'caseid'])
)
print(f'Total cases found: {len(caseids)}')

Total cases found: 981


We found **981 cases** with cardiac index monitoring — about 15% of the full VitalDB cohort. This makes sense: cardiac output monitors are used selectively, not routinely. These 981 patients form our study population.

Next, we load the full case demographics and laboratory results so we can match each patient's intraoperative cardiac index data with their pre- and postoperative creatinine values.

In [8]:
df_cases = pd.read_csv("vital_db/all_cases.csv")
df_labs = pd.read_csv("vital_db/labs.csv")

## Linking cardiac index to kidney injury: the main analysis

Now comes the heart of this notebook. For each of our 981 patients, we need to do four things:

1. **Find the preoperative creatinine** — the last creatinine value measured *before* surgery begins. This is our baseline kidney function.
2. **Find the postoperative creatinine** — the highest creatinine value within 48 hours *after* surgery ends. If this is more than 1.5 times the baseline, we classify the patient as having AKI (KDIGO Stage 1).
3. **Load the intraoperative cardiac index** — the continuous CI signal recorded by the cardiac output monitor during surgery.
4. **Calculate exposure to low CI** — for a range of CI thresholds (from 0 to 4 L/min/m²), we compute what percentage of the surgical time the patient spent *below* each threshold.

### Why "time below threshold"?

Think of it this way: a brief dip in cardiac output might be harmless, but spending a large proportion of a long operation with inadequate cardiac output could starve the kidneys of the blood flow they need. By calculating the percentage of time below various thresholds, we can later ask: "Is spending more time at low CI associated with a greater risk of AKI?"

This approach is sometimes called a **dose-response analysis** — we are looking at whether more "exposure" to low CI leads to more kidney injury.

> **Note:** This cell loads cardiac index data from the VitalDB API for each case, which may take several minutes to run. The results are printed case by case so you can follow the progress.

In [9]:
# Set blood pressure threshold
thresholds = np.arange(0, 4, 0.1)

# Save the final result
rows = []
for caseid in caseids:
    print('loading {}...'.format(caseid), flush=True, end='')

    # Column ['anend'] : anesthesia end time
    aneend = df_cases[(df_cases['caseid'] == caseid)]['aneend'].values[0]

    # Last creatinine concentration before surgery
    preop_cr = df_labs[(df_labs['caseid'] == caseid) & (df_labs['dt'] < 0) & (df_labs['name'] == 'cr')].sort_values(by=['dt'], axis=0, ascending=False)['result'].values.flatten()
    if len(preop_cr) == 0:
        print('no preop cr')
        continue
    preop_cr = preop_cr[0]

    # Maximum creatinine concentration within 48 hours after surgery
    postop_cr = df_labs[(df_labs['caseid'] == caseid) & (df_labs['dt'] > aneend) &
        (df_labs['dt'] < aneend + 48 * 3600) & (df_labs['name'] == 'cr')]['result'].max(skipna=True)
    if not postop_cr or np.isnan(postop_cr):
        print('no postop cr')
        continue

    # KDIGO stage I
    aki = postop_cr > preop_cr * 1.5

    # Blood pressure during surgery
    cis = vitaldb.load_case(caseid, '/CI').flatten()
    cis = cis[~np.isnan(cis)]
    cis = cis[(cis > 0.1) & (cis < 4)]
    if len(cis) < 10:
        print('no ci')
        continue

    # Calculate the percentage that stays for the time as increasing the blood pressure by 1 unit.
    row = {'aki':aki}
    for threshold in thresholds:
        row[f'under{threshold}'] = np.nanmean(cis < threshold) * 100

    # Append the result into row
    rows.append(row)

    print(f'{preop_cr} -> {postop_cr}, {"AKI" if aki else "no AKI"}, min CI={min(cis)}, max CI={max(cis)}')

df = pd.DataFrame(rows)
print(f'{df["aki"].sum()} AKI/ {len(df)} cases {df["aki"].mean() * 100:.1f}%')

loading 4098...0.84 -> 1.14, no AKI, min CI=0.10000000149011612, max CI=3.9000000953674316
loading 6147...no preop cr
loading 2057...1.23 -> 1.35, no AKI, min CI=1.2999999523162842, max CI=3.200000047683716
loading 12...0.58 -> 0.65, no AKI, min CI=3.299999952316284, max CI=3.9000000953674316
loading 6160...no postop cr
loading 17...1.05 -> 1.09, no AKI, min CI=2.0999999046325684, max CI=3.9000000953674316
loading 4112...no postop cr
loading 19...1.19 -> 0.78, no AKI, min CI=0.10000000149011612, max CI=3.9000000953674316
loading 6163...0.77 -> 0.76, no AKI, min CI=0.4000000059604645, max CI=3.9000000953674316
loading 22...no preop cr
loading 4120...1.03 -> 0.96, no AKI, min CI=2.9000000953674316, max CI=3.9000000953674316
loading 2072...no postop cr
loading 26...0.73 -> 0.78, no AKI, min CI=1.5, max CI=3.5999999046325684
loading 25...no preop cr
loading 29...0.78 -> 0.74, no AKI, min CI=1.0, max CI=3.0999999046325684
loading 2082...1.46 -> 1.29, no AKI, min CI=2.0999999046325684, max C

## What did we find?

Out of our 981 cases with cardiac output monitoring, **783 had both pre- and postoperative creatinine** values available — the rest were excluded because one or both measurements were missing, or because the CI recording was too short to be meaningful.

Of these 783 patients, **34 developed AKI** — an incidence of **4.3%**. This is within the expected range for a mixed surgical population, though somewhat lower than studies focusing exclusively on high-risk cardiac or major abdominal surgery.

### What do the numbers tell us?

For each patient in the output above, you can see:
- The **preoperative creatinine** → **postoperative creatinine** (e.g., `0.8 -> 0.9`)
- Whether they developed **AKI** or **no AKI** (based on the 1.5× threshold)
- The **minimum and maximum cardiac index** recorded during their operation

The dataset we have built (`df`) now contains, for each patient, their AKI status and the percentage of surgical time spent below every CI threshold from 0 to 4 L/min/m² (in steps of 0.1). This is the foundation for exploring the relationship between low cardiac output and kidney injury.

### Limitations to keep in mind

A few important caveats before we draw conclusions:

- **Selection bias:** Patients who received cardiac output monitoring were likely sicker or undergoing more complex surgery than the average VitalDB patient. Our 4.3% AKI rate applies to *this* population, not to all surgical patients.
- **Observational data:** We are observing associations, not proving causation. Patients with low CI may also have had low blood pressure, received nephrotoxic drugs, or had pre-existing kidney disease — any of which could contribute to AKI independently.
- **KDIGO Stage 1 only:** We used the mildest AKI definition (creatinine >1.5× baseline). More severe stages (2× or 3× elevation, or need for dialysis) would yield even fewer cases but perhaps stronger associations.

In the companion notebooks (**Chapter 9.1** and **9.2**), we extend this analysis by examining blood pressure thresholds alongside cardiac index, and by building predictive models that account for multiple risk factors simultaneously.